# Mangrove Fragmentation Segmentation, 1990–2024

Input: binary mangrove classifications (1 = mangrove, 0 = non-mangrove, ~30 m, EPSG:4326), one GeoTIFF per year.

**1. Fragmentation segmentation** (Landscape Fragmentation Tool, Vogt et al. 2007; Parent & Hurd, LFT v2). Each mangrove pixel is labelled:

| Class | Rule |
|---|---|
| **Patch** | fragment too small or thin to hold any core |
| **Edge** | within `EDGE_WIDTH_M` of the outer non-mangrove boundary |
| **Perforated** | within `EDGE_WIDTH_M` of a small enclosed opening (< `GAP_MAX_HA`) |
| **Small / Medium / Large core** | farther than `EDGE_WIDTH_M` from any non-mangrove; split by contiguous core size (< 100 ha, 100–200 ha, > 200 ha) |

**2. Landscape metrics** computed with `pylandstats` and named like R **`landscapemetrics`** (`lsm_c_ca`, `lsm_c_np`, `lsm_c_ed`, …), with the same conventions: 8-neighbour patches, landscape boundary not counted as edge (`count_boundary = FALSE` / `consider_boundary = FALSE`), area in ha, ED in m/ha, PD per 100 ha.

**How to run:** set the folders and parameters below, then run all cells top to bottom. Each processing step shows a progress bar, and a cell refuses to run until the cells it depends on have run with the current parameters. Figures are saved as PNG files in `OUT_DIR/figures/`, the interactive map (MapLibre GL) and Sankey diagrams as HTML in `OUT_DIR/interactive/`; the notebook shows them from those files (nothing is embedded). The last cell exports GeoTIFFs and tables from the results already in memory. The same pipeline runs without Jupyter: `python src/mangfrag.py`.

## Parameters

In [1]:
from pathlib import Path

# Folders (relative to this notebook, or absolute)
DATA_DIR = Path("data/mangrove_1990-2024")  # input: one classification GeoTIFF per year, year in the file name
OUT_DIR = Path("outputs")               # output: figures/, rasters/, tables/, interactive/

# Segmentation (defaults follow LFT, converted to metric)
EDGE_WIDTH_M = 100      # edge depth (m); also used as core edge_depth for tca / ndca
GAP_MAX_HA = 5          # enclosed openings smaller than this are 'perforations'
CORE_SMALL_HA = 100     # core < this = small core
CORE_LARGE_HA = 200     # core >= this = large core

# Metrics
ENN = True              # mean nearest-neighbour distance (~2 min total, parallel); False to iterate faster

In [2]:
import os
import sys
import matplotlib
matplotlib.use("Agg")  # figures go to files only, never embedded in the notebook
import pandas as pd
from IPython.display import HTML, IFrame, display
sys.path.insert(0, str(Path("src").resolve()))  # mangfrag.py lives in src/
import mangfrag as mf

pd.set_option("display.float_format", "{:,.3f}".format)
pd.set_option("display.max_columns", 30)


class StepNotRun(Exception):
    """Raised when a cell needs an earlier cell; shown as one red line instead of a traceback."""
    def _render_traceback_(self):
        return [f"\x1b[1;31m⛔ {self}\x1b[0m"]


def _seg_params():
    return dict(edge_width_m=EDGE_WIDTH_M, gap_max_ha=GAP_MAX_HA, core_small_ha=CORE_SMALL_HA, core_large_ha=CORE_LARGE_HA)


# step: (cell to run, check that returns None if ok, or why it must be (re-)run)
_STEPS = {
    "load": ("1.1 Load data", lambda r: "not run yet" if r is None else
             "DATA_DIR changed" if r.data_dir != Path(DATA_DIR).resolve() else None),
    "segmentation": ("1.2 Fragmentation segmentation", lambda r: "not run yet" if not hasattr(r, "seg_params") else
                     "parameters changed" if r.seg_params != _seg_params() else None),
    "metrics": ("1.3 Landscape metrics", lambda r: "not run yet" if not hasattr(r, "metric_params") else
                "parameters changed" if r.metric_params != dict(**_seg_params(), enn=ENN) else None),
}


def require(*steps):
    """Stop this cell unless the given steps have run with the current parameters."""
    r = globals().get("res")
    for step in steps:
        cell, check = _STEPS[step]
        why = check(r)
        if why:
            raise StepNotRun(f"Run the '{cell}' cell first ({why}).")


if "DATA_DIR" not in globals():
    raise StepNotRun("Run the 'Parameters' cell first.")


def _rel(path):
    return Path(os.path.relpath(path)).as_posix()


def show(figs):
    """{name: lambda returning a figure}: render + save each to OUT_DIR/figures (with a progress bar),
    then show them side by side in one row, by file reference (no embedded images)."""
    imgs = "".join(f'<img src="{_rel(p)}" alt="{p.stem}" style="flex:1 1 380px;min-width:0;max-width:100%">'
                   for p in mf.save_figures(figs, OUT_DIR))
    display(HTML(f'<div style="display:flex;flex-wrap:wrap;gap:12px;align-items:flex-start">{imgs}</div>'))


def embed(path, height):
    """Show a saved interactive HTML file (map, Sankey) inside the cell output."""
    display(IFrame(_rel(path), width="100%", height=height))

# 1 · Processing
Run these three cells in order. Everything below reads their results.

### 1.1 Load data

In [3]:
res = mf.load(DATA_DIR)  # starts fresh: later steps must run again
print(f"Years: {res.years}")
print(f"Grid: {res.profile['width']} x {res.profile['height']} px · pixel {res.px:.1f} m x {res.py:.1f} m ({res.pix_ha:.4f} ha)")
print(f"Landscape area: {res.masks[res.years[0]].size * res.pix_ha:,.0f} ha")

Loading rasters:   0%|          | 0/8 [00:00<?, ?file/s]

Years: [1990, 1995, 2000, 2005, 2010, 2015, 2020, 2024]
Grid: 2233 x 2661 px · pixel 30.0 m x 29.8 m (0.0894 ha)
Landscape area: 531,174 ha


### 1.2 Fragmentation segmentation

In [4]:
require("load")
res = mf.run_segmentation(res, **_seg_params())

Segmentation:   0%|          | 0/8 [00:00<?, ?year/s]

### 1.3 Landscape metrics (mangrove class)
Runs one process per year. Takes about 2 minutes with `ENN = True`.

In [5]:
require("load", "segmentation")
res = mf.run_metrics(res, enn=ENN)

Landscape metrics:   0%|          | 0/8 [00:00<?, ?year/s]

# 2 · Statistics & visuals
Figures are saved to `OUT_DIR/figures/`, interactive HTML to `OUT_DIR/interactive/`, and shown from there.

### 2.1 Fragmentation maps

In [6]:
require("load", "segmentation")
show({"maps_all_years": lambda: mf.plot_map_grid(res.segs, res.extent)})

Figures:   0%|          | 0/2 [00:00<?, ?step/s]

In [7]:
require("load", "segmentation")
y0, y1 = res.years[0], res.years[-1]
show({f"map_{y0}": lambda: mf.plot_map(res.segs[y0], y0, res.extent),
      f"map_{y1}": lambda: mf.plot_map(res.segs[y1], y1, res.extent)})

Figures:   0%|          | 0/4 [00:00<?, ?step/s]

### 2.2 Interactive map
MapLibre GL: switch years, overlay the change layer, adjust opacity, change basemap (satellite / OSM). Needs internet for the basemap and the MapLibre library.

In [9]:
require("load", "segmentation")
html = mf.export_interactive(res, OUT_DIR, only=["fragmentation_map"])
embed(html[0], 640)

Interactive HTML:   0%|          | 0/1 [00:00<?, ?file/s]

### 2.3 Class composition

In [10]:
require("load", "segmentation")
area = res.composition
share = area.drop(columns="Non-mangrove").pipe(lambda d: d.div(d.sum(axis=1), axis=0) * 100)
display(area.style.format("{:,.0f}").set_caption("Area (ha)"))
display(share.style.format("{:.1f}").background_gradient(cmap="Greens", axis=None).set_caption("Share of mangrove area (%)"))
show({"composition_ha": lambda: mf.plot_composition(res.composition),
      "composition_pct": lambda: mf.plot_composition(res.composition, percent=True)})

,Non-mangrove,Patch,Edge,Perforated,Small core,Medium core,Large core
year,,,,,,,
1990,"426,109",951,"25,105","9,556","3,903","1,950","63,601"
1995,"428,422","1,254","24,908","9,225","3,814","2,859","60,691"
2000,"459,456","3,593","26,648","11,913","9,301","2,964","17,300"
2005,"477,439","4,529","21,577","7,115","7,330","2,340","10,843"
2010,"477,273","4,552","21,878","5,881","7,615","1,529","12,446"
2015,"476,166","4,506","22,154","5,530","7,424","1,971","13,422"
2020,"478,901","4,587","20,756","4,614","6,749","3,338","12,229"
2024,"477,112","4,546","21,811","5,582","6,834","3,404","11,885"


,Patch,Edge,Perforated,Small core,Medium core,Large core
year,,,,,,
1990,0.9,23.9,9.1,3.7,1.9,60.5
1995,1.2,24.2,9.0,3.7,2.8,59.1
2000,5.0,37.2,16.6,13.0,4.1,24.1
2005,8.4,40.2,13.2,13.6,4.4,20.2
2010,8.4,40.6,10.9,14.1,2.8,23.1
2015,8.2,40.3,10.1,13.5,3.6,24.4
2020,8.8,39.7,8.8,12.9,6.4,23.4
2024,8.4,40.3,10.3,12.6,6.3,22.0


Figures:   0%|          | 0/4 [00:00<?, ?step/s]

### 2.4 Landscape metrics
Column names match `landscapemetrics::lsm_c_<name>`. Definitions and units:

In [11]:
info = pd.DataFrame(mf.METRIC_INFO, index=["description", "unit"]).T
info.index = "lsm_c_" + info.index
info

,description,unit
lsm_c_ca,Total class area,ha
lsm_c_pland,Percentage of landscape,%
lsm_c_np,Number of patches,n
lsm_c_pd,Patch density,n / 100 ha
lsm_c_lpi,Largest patch index,%
lsm_c_te,Total edge,m
lsm_c_ed,Edge density,m / ha
lsm_c_lsi,Landscape shape index,-
lsm_c_area_mn,Mean patch area,ha
lsm_c_area_md,Median patch area,ha


In [12]:
require("load", "segmentation", "metrics")
display(res.metrics.T.style.format("{:,.3f}").set_caption("Class-level metrics per year"))
change = (res.metrics.iloc[-1] / res.metrics.iloc[0] - 1) * 100
display(change.rename(f"% change {res.years[0]}→{res.years[-1]}").to_frame().style.format("{:+.1f}"))
show({"landscape_metrics": lambda: mf.plot_metrics(res.metrics)})

year,1990,1995,2000,2005,2010,2015,2020,2024
ca,"105,064.585","102,751.366","71,717.742","53,734.671","53,900.674","55,007.089","52,272.383","54,061.849"
pland,19.780,19.344,13.502,10.116,10.147,10.356,9.841,10.178
np,739.000,"1,201.000","2,790.000","2,490.000","2,450.000","2,350.000","2,442.000","2,164.000"
pd,0.139,0.226,0.525,0.469,0.461,0.442,0.460,0.407
lpi,1.882,1.770,0.978,0.754,0.789,0.802,0.744,0.776
te,"6,184,738.106","6,513,685.975","9,235,154.336","7,830,280.604","7,493,224.418","7,417,557.396","6,979,735.431","7,433,637.967"
ed,11.644,12.263,17.386,14.741,14.107,13.964,13.140,13.995
lsi,47.698,50.796,86.208,84.446,80.686,79.061,76.316,79.922
area_mn,142.171,85.555,25.705,21.580,22.000,23.407,21.406,24.982
area_md,0.894,0.268,0.715,1.430,1.520,1.698,1.609,1.877


,% change 1990→2024
ca,-48.5
pland,-48.5
np,+192.8
pd,+192.8
lpi,-58.8
te,+20.2
ed,+20.2
lsi,+67.6
area_mn,-82.4
area_md,+110.0


Figures:   0%|          | 0/2 [00:00<?, ?step/s]

### 2.5 Patch size distribution

In [13]:
require("load", "segmentation", "metrics")
display(res.patches.groupby("year")["area"].describe().style.format("{:,.2f}").set_caption("Patch area (ha)"))
show({"patch_sizes": lambda: mf.plot_patch_sizes(res.patches)})

,count,mean,std,min,25%,50%,75%,max
year,,,,,,,,
1990,739.00,142.17,816.06,0.09,0.18,0.89,3.22,"9,995.19"
1995,"1,201.00",85.55,581.44,0.09,0.09,0.27,1.88,"9,399.57"
2000,"2,790.00",25.71,211.28,0.09,0.09,0.72,2.41,"5,197.21"
2005,"2,490.00",21.58,162.69,0.09,0.72,1.43,4.47,"4,005.43"
2010,"2,450.00",22.00,161.01,0.09,0.72,1.52,4.74,"4,191.54"
2015,"2,350.00",23.41,162.10,0.09,0.80,1.70,5.10,"4,262.61"
2020,"2,442.00",21.41,151.95,0.09,0.80,1.61,4.65,"3,952.06"
2024,"2,164.00",24.98,167.65,0.09,0.89,1.88,5.63,"4,121.64"


Figures:   0%|          | 0/2 [00:00<?, ?step/s]

### 2.6 Change and class transitions

In [14]:
require("load", "segmentation")
y0, y1 = res.years[0], res.years[-1]
show({f"change_{y0}_{y1}": lambda: mf.plot_change(res.masks[y0], res.masks[y1], y0, y1, res.extent, res.pix_ha)})
t = mf.transitions(res.segs[y0], res.segs[y1], res.pix_ha)
t.style.format("{:,.0f}").background_gradient(cmap="Blues", axis=None).set_caption(f"Transition matrix {y0} → {y1} (ha)")

Figures:   0%|          | 0/2 [00:00<?, ?step/s]

to,Non-mangrove,Patch,Edge,Perforated,Small core,Medium core,Large core
from,,,,,,,
Non-mangrove,"422,577",427,"1,800",355,496,80,373
Patch,481,226,197,20,15,0,12
Edge,"11,693","1,704","10,192",738,386,101,291
Perforated,"4,670",340,"1,527","1,985",383,159,491
Small core,"1,242",76,398,153,"1,533",341,160
Medium core,889,50,241,68,149,246,306
Large core,"35,559","1,722","7,456","2,262","3,871","2,478","10,253"


### 2.7 Sankey diagrams
Hover a flow or node for hectares. Every column sums to the same area (pixels that were mangrove in at least one year).

In [15]:
require("load", "segmentation")
y0, y1 = res.years[0], res.years[-1]
html = mf.export_interactive(res, OUT_DIR, only=[f"sankey_{y0}_{y1}"])
embed(html[0], 760)

Interactive HTML:   0%|          | 0/1 [00:00<?, ?file/s]

In [16]:
require("load", "segmentation")
html = mf.export_interactive(res, OUT_DIR, only=["sankey_all_years"])
embed(html[0], 760)

Interactive HTML:   0%|          | 0/1 [00:00<?, ?file/s]

# 3 · Export
Writes GeoTIFFs (with embedded colour table, open directly in QGIS/ArcGIS) and CSV + multi-sheet Excel (all transition matrices) from the results computed above. Figures and interactive HTML were already saved by section 2.

In [17]:
require("load", "segmentation", "metrics")
mf.export_rasters(res, OUT_DIR)
mf.export_tables(res, OUT_DIR)
for f in sorted(OUT_DIR.rglob("*.*")):
    print(f)

GeoTIFFs:   0%|          | 0/8 [00:00<?, ?year/s]

Tables:   0%|          | 0/18 [00:00<?, ?table/s]

outputs/figures/change_1990_2024.png
outputs/figures/composition_ha.png
outputs/figures/composition_pct.png
outputs/figures/landscape_metrics.png
outputs/figures/map_1990.png
outputs/figures/map_2024.png
outputs/figures/maps_all_years.png
outputs/figures/patch_sizes.png
outputs/interactive/fragmentation_map.html
outputs/interactive/sankey_1990_2024.html
outputs/interactive/sankey_all_years.html
outputs/rasters/change_1990_2024.tif
outputs/rasters/fragmentation_1990.tif
outputs/rasters/fragmentation_1995.tif
outputs/rasters/fragmentation_2000.tif
outputs/rasters/fragmentation_2005.tif
outputs/rasters/fragmentation_2010.tif
outputs/rasters/fragmentation_2015.tif
outputs/rasters/fragmentation_2020.tif
outputs/rasters/fragmentation_2024.tif
outputs/tables/fragmentation_area_ha.csv
outputs/tables/fragmentation_share_pct.csv
outputs/tables/landscape_metrics.csv
outputs/tables/mangrove_fragmentation.xlsx
outputs/tables/patches.csv
